In [1]:
# 0.0 — Setup
from pathlib import Path
import json, math, re, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Dossiers racine à explorer
SEARCH_DIRS = [
    Path("."), Path("./reviewer_pack"), Path("./outputs"), Path("./NN"),
    Path("/home/a.riyahi/spinn_project"),
    Path("/home/a.riyahi/spinn_project/NN"),
    Path("/home/a.riyahi/spinn_project/reviewer_pack"),
    Path("/home/a.riyahi/spinn_project/outputs"),
]

OUT_DIR = Path("/home/a.riyahi/spinn_project/outputs")
FIG_DIR = OUT_DIR / "figs"
TAB_DIR = OUT_DIR / "tables"
PKG_DIR = Path("reviewer_pack")
for d in [OUT_DIR, FIG_DIR, TAB_DIR, PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def latest(patterns, search_dirs=SEARCH_DIRS):
    if isinstance(patterns, str): patterns = [patterns]
    hits = []
    for d in search_dirs:
        if not d.exists(): continue
        for pat in patterns:
            hits += list(d.rglob(pat))
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def list_all(patterns, search_dirs=SEARCH_DIRS, limit=50):
    if isinstance(patterns, str): patterns = [patterns]
    hits = []
    for d in search_dirs:
        if not d.exists(): continue
        for pat in patterns:
            hits += list(d.rglob(pat))
    hits = sorted(hits, key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[:limit]

In [2]:
# 1.0 — Localiser les artefacts NN
nn_metrics = latest(["metrics_comparison_rounded*.csv","metrics_comparison*.csv"])
nn_perclass = latest(["metrics_per_class_rounded*.csv","metrics_per_class*.csv"])
nn_fn = latest(["false_negatives*.csv"])
nn_histories = list_all(["**/history*.npz"])
nn_preds = {
    "NN V1": latest(["**/predictions_NN_V1*.npz","**/predictions*V1*.npz"]),
    "NN V2": latest(["**/predictions_NN_V2*.npz","**/predictions*V2*.npz"]),
    "NN V3": latest(["**/predictions_NN_V3*.npz","**/predictions*V3*.npz"]),
}

print("metrics:", nn_metrics)
print("per-class:", nn_perclass)
print("false_negatives:", nn_fn)
print("history files (<=5):", nn_histories[:5])
print("predictions:", nn_preds)


metrics: NN/reviewer_pack/metrics_comparison_rounded.csv
per-class: NN/reviewer_pack/metrics_per_class_rounded.csv
false_negatives: NN/reviewer_pack/false_negatives.csv
history files (<=5): [PosixPath('NN/outputs/NN_V1/history.npz'), PosixPath('NN/outputs/NN_V1/history.npz'), PosixPath('/home/a.riyahi/spinn_project/NN/outputs/NN_V1/history.npz'), PosixPath('/home/a.riyahi/spinn_project/NN/outputs/NN_V1/history.npz'), PosixPath('NN/outputs/NN_V2/history.npz')]
predictions: {'NN V1': None, 'NN V2': None, 'NN V3': None}


In [3]:
# 2.0 — Charger / harmoniser récap NN
def to_pct(series):
    s = series.astype(float)
    return (100*s).round(2) if s.max()<=1 else s.round(2)

if nn_metrics is None:
    raise FileNotFoundError("metrics_comparison*.csv introuvable. Génère-le depuis tes notebooks NN.")
dfm = pd.read_csv(nn_metrics)

# Harmoniser les noms
colmap = {
    'model':'Model','Model':'Model',
    'accuracy':'Val_Accuracy_(%)','val_accuracy':'Val_Accuracy_(%)','val_acc':'Val_Accuracy_(%)',
    'macro_f1':'Macro_F1_(%)','f1_macro':'Macro_F1_(%)',
    'macro_auc':'ROC_AUC_macro','roc_auc_macro':'ROC_AUC_macro'
}
for c in list(dfm.columns):
    if c in colmap: dfm = dfm.rename(columns={c: colmap[c]})

if 'Val_Accuracy_(%)' in dfm: dfm['Val_Accuracy_(%)'] = to_pct(dfm['Val_Accuracy_(%)'])
if 'Macro_F1_(%)'    in dfm: dfm['Macro_F1_(%)']    = to_pct(dfm['Macro_F1_(%)'])

# Restreindre aux NN V1/V2/V3
def norm_key(s): return "".join(ch for ch in str(s).upper() if ch.isalnum())
dfm['key'] = dfm['Model'].map(norm_key)
want = {'NNV1':'NN V1','NNV2':'NN V2','NNV3':'NN V3'}
rows = []
for k,label in want.items():
    sub = dfm[dfm['key']==k].head(1)
    rec = {'Model': label,
           'Val_Accuracy_(%)': np.nan,
           'Macro_F1_(%)': np.nan,
           'ROC_AUC_macro': np.nan,
           'False_Negatives': np.nan,
           'Notes':'data-driven'}
    if len(sub):
        for c in ['Val_Accuracy_(%)','Macro_F1_(%)','ROC_AUC_macro']:
            if c in sub.columns: rec[c] = sub.iloc[0][c]
    rows.append(rec)

# False negatives si dispo
if nn_fn is not None:
    dff = pd.read_csv(nn_fn)
    # chercher colonnes modèle + FN
    mod_col = next((c for c in dff.columns if c.lower() in ['model','name']), None)
    fn_col  = next((c for c in dff.columns if 'fn' in c.lower()), None)
    if mod_col and fn_col:
        dff['key'] = dff[mod_col].map(norm_key)
        for k,label in want.items():
            hit = dff[dff['key']==k]
            if len(hit):
                for r in rows:
                    if r['Model']==label: r['False_Negatives'] = int(hit.iloc[0][fn_col])

df_nn = pd.DataFrame(rows)

# AUC macro à partir des prédictions si manquante
def auc_from_npz(path):
    D = np.load(path)
    y_true = D.get('y_true_test', D.get('y_true'))
    y_prob = D.get('y_prob_test', D.get('y_prob'))
    if y_true is None or y_prob is None: return np.nan
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob)
    if y_prob.ndim==1 or (y_prob.ndim==2 and y_prob.shape[1]==1):
        return float(roc_auc_score(y_true, y_prob.ravel()))
    return float(roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'))

updated = False
for m in ["NN V1","NN V2","NN V3"]:
    if pd.isna(df_nn.loc[df_nn.Model==m, "ROC_AUC_macro"]).all():
        p = nn_preds.get(m)
        if p and p.exists():
            auc = auc_from_npz(p)
            if not np.isnan(auc):
                df_nn.loc[df_nn.Model==m, "ROC_AUC_macro"] = round(auc, 3)
                updated = True

print("NN summary:")
display(df_nn)
df_nn.to_csv(PKG_DIR/"nn_summary_for_table1.csv", index=False)


NN summary:


,Model,Val_Accuracy_(%),Macro_F1_(%),ROC_AUC_macro,False_Negatives,Notes
0,NN V1,97.71,97.76,NaN,0,data-driven
1,NN V2,97.25,97.14,NaN,0,data-driven
2,NN V3,97.71,97.60,NaN,0,data-driven


In [4]:
# 3.0 — Charger KPIs PINN
pinn_row_csv = latest(["table1_pinn_row*.csv"])
pinn_kpi_json = latest(["pinn_kpis_*.json"])
pinn_diag_npz = latest(["pinn_diag_*.npz"])

def build_pinn_row():
    if pinn_row_csv:
        df = pd.read_csv(pinn_row_csv)
        df['Model'] = df.get('Model','PINN V1')
        return df.head(1)
    if pinn_kpi_json:
        k = json.loads(pinn_kpi_json.read_text())
        return pd.DataFrame([{
            "Model":"PINN V1",
            "Val_Accuracy_(%)": np.nan,
            "Macro_F1_(%)":     np.nan,
            "ROC_AUC_macro":    np.nan,
            "False_Negatives":  np.nan,
            "PINN_L2_mean(h)":  k.get("h_L2_mean", np.nan),
            "PINN_abs_h_p95":   k.get("abs_h_p95", np.nan),
            "omega0_rad_s":     k.get("omega0_rad_s", np.nan),
            "omega_est_rad_s":  k.get("omega_est_rad_s", np.nan),
            "abs_rel_err_omega(%)": k.get("abs_rel_err_omega", np.nan),
            "Notes":"physics residual; SDOF consistency"
        }])
    if pinn_diag_npz:
        with np.load(pinn_diag_npz) as D:
            h = D.get("h") or D.get("h_vals") or D.get("residual")
            abs_h = np.abs(h) if h is not None else None
            l2 = float(np.mean(h**2)) if h is not None else np.nan
            p95 = float(np.percentile(abs_h,95)) if abs_h is not None else np.nan
            w0  = D.get("omega0") or D.get("omega0_rad_s")
            we  = D.get("omega_est") or D.get("omega_est_rad_s")
            w0  = float(np.squeeze(w0)) if w0 is not None else np.nan
            we  = float(np.squeeze(we)) if we is not None else np.nan
            rel = (abs(we-w0)/w0*100.0) if (w0 and w0!=0 and not np.isnan(we)) else np.nan
        return pd.DataFrame([{
            "Model":"PINN V1","Val_Accuracy_(%)":np.nan,"Macro_F1_(%)":np.nan,"ROC_AUC_macro":np.nan,"False_Negatives":np.nan,
            "PINN_L2_mean(h)":l2,"PINN_abs_h_p95":p95,"omega0_rad_s":w0,"omega_est_rad_s":we,"abs_rel_err_omega(%)":rel,
            "Notes":"physics residual; SDOF consistency"
        }])
    raise RuntimeError("Aucun KPI PINN trouvé (table1_pinn_row*.csv / pinn_kpis_*.json / pinn_diag_*.npz).")

df_pinn = build_pinn_row()
display(df_pinn)


,Model,Val_Accuracy_(%),Macro_F1_(%),ROC_AUC_macro,False_Negatives,PINN_L2_mean(h),PINN_abs_h_p95,omega0_rad_s,omega_est_rad_s,abs_rel_err_omega(%),Notes
0,PINN V1,NaN,NaN,NaN,NaN,0.001205,0.0666,25.132742,43.181559,71.81,physics residual; SDOF consistency


In [5]:
# 4.0 — Fusion & exports
cols = ["Model","Val_Accuracy_(%)","Macro_F1_(%)","ROC_AUC_macro","False_Negatives",
        "PINN_L2_mean(h)","PINN_abs_h_p95","omega0_rad_s","omega_est_rad_s","abs_rel_err_omega(%)","Notes"]

for c in cols:
    if c not in df_nn.columns: df_nn[c]=np.nan
    if c not in df_pinn.columns: df_pinn[c]=np.nan

df_t1 = pd.concat([df_nn[cols], df_pinn[cols]], ignore_index=True)

# Mise en forme "publication"
def fmt_pct(v):  return "" if pd.isna(v) else f"{float(v):.2f}"
def fmt_auc(v):  return "" if pd.isna(v) else f"{float(v):.3f}"
def fmt_sci(v):  return "" if pd.isna(v) else (f"{v:.3e}" if (abs(v)<1e-2 or abs(v)>=1e3) else f"{v:.5f}")
def fmt_f(v):    return "" if pd.isna(v) else f"{float(v):.6f}"

df_pub = df_t1.copy()
for c in ["Val_Accuracy_(%)","Macro_F1_(%)","abs_rel_err_omega(%)"]:
    df_pub[c] = df_pub[c].map(fmt_pct)
if "ROC_AUC_macro" in df_pub: df_pub["ROC_AUC_macro"] = df_pub["ROC_AUC_macro"].map(fmt_auc)
for c in ["PINN_L2_mean(h)","PINN_abs_h_p95"]:
    df_pub[c] = df_pub[c].map(fmt_sci)
for c in ["omega0_rad_s","omega_est_rad_s"]:
    df_pub[c] = df_pub[c].map(fmt_f)
if "False_Negatives" in df_pub:
    df_pub["False_Negatives"] = df_t1["False_Negatives"].apply(lambda x: "" if pd.isna(x) else int(x))

csv_path = TAB_DIR / "table1_summary.csv"
md_path  = TAB_DIR / "table1_summary.md"
df_pub.to_csv(csv_path, index=False)
md_path.write_text(df_pub.to_markdown(index=False), encoding="utf-8")

print("→ Table 1 CSV:", csv_path)
print("→ Table 1 MD :", md_path)
display(df_pub)


→ Table 1 CSV: /home/a.riyahi/spinn_project/outputs/tables/table1_summary.csv
→ Table 1 MD : /home/a.riyahi/spinn_project/outputs/tables/table1_summary.md


,Model,Val_Accuracy_(%),Macro_F1_(%),ROC_AUC_macro,False_Negatives,PINN_L2_mean(h),PINN_abs_h_p95,omega0_rad_s,omega_est_rad_s,abs_rel_err_omega(%),Notes
0,NN V1,97.71,97.76,,0,,,,,,data-driven
1,NN V2,97.25,97.14,,0,,,,,,data-driven
2,NN V3,97.71,97.60,,0,,,,,,data-driven
3,PINN V1,,,,,1.205e-03,0.06660,25.132742,43.181559,71.81,physics residual; SDOF consistency


In [6]:
# 5.0 — Courbes train/val (robuste, 1 figure verticale par modèle : Accuracy + Loss)
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def safe_get(H, names):
    """Retourne la première clé trouvée dans 'names' sous forme d'array 1D float, sinon None."""
    for n in names:
        if n in H.files:
            arr = np.array(H[n]).astype(float).ravel()
            return arr
    return None

def find_history_for_model(model_label):
    """Trouve le history*.npz le plus récent qui correspond au modèle (NN V1/V2/V3)."""
    tags_by_model = {
        "NN V1": ["NN_V1", "V1"],
        "NN V2": ["NN_V2", "V2"],
        "NN V3": ["NN_V3", "V3"],
    }
    tags = tags_by_model[model_label]
    candidates = []
    for d in SEARCH_DIRS:
        if not d.exists(): 
            continue
        for pat in ["history*.npz", "**/history*.npz"]:
            for p in d.rglob(pat):
                s = str(p).lower().replace(" ", "")
                if any(t.lower() in s for t in tags):
                    candidates.append(p)
    if not candidates:
        # dernier recours : premier history connu, s'il existe
        return nn_histories[0] if nn_histories else None
    return max(candidates, key=lambda p: p.stat().st_mtime)

def plot_curves_stacked(hist_path, model_label, out_prefix, lw=1.9):
    H = np.load(hist_path)
    tr_acc = safe_get(H, ["accuracy","acc","train_accuracy"])
    va_acc = safe_get(H, ["val_accuracy","val_acc"])
    tr_loss = safe_get(H, ["loss","train_loss"])
    va_loss = safe_get(H, ["val_loss"])

    # nombre d'époques détecté
    L = max([len(x) for x in [tr_acc, va_acc, tr_loss, va_loss] if x is not None])
    epochs = np.arange(1, L+1)

    # figure verticale compacte (format colonne)
    fig, axes = plt.subplots(2, 1, figsize=(3.6, 4.8), dpi=300, constrained_layout=True)

    # Accuracy (Train en C0, Val en C1)
    ax = axes[0]
    if tr_acc is not None: ax.plot(epochs[:len(tr_acc)], tr_acc, label="Train", linewidth=lw)
    if va_acc is not None: ax.plot(epochs[:len(va_acc)], va_acc, label="Val", linewidth=lw, linestyle="--")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=7)
    ax.set_title(f"{model_label} — Accuracy", fontsize=10)

    # Loss (Train en C0, Val en C1)
    ax = axes[1]
    if tr_loss is not None: ax.plot(epochs[:len(tr_loss)], tr_loss, label="Train", linewidth=lw)
    if va_loss is not None: ax.plot(epochs[:len(va_loss)], va_loss, label="Val", linewidth=lw, linestyle="--")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(fontsize=7)
    ax.set_title(f"{model_label} — Loss", fontsize=10)

    # export
    outfile = FIG_DIR / f"{out_prefix}_curves.png"
    plt.savefig(outfile)
    plt.close(fig)
    print(f"[OK] {model_label} → {outfile}")

# Générer les 3 figures (une par modèle)
for m, tag in [("NN V1","nn_v1"), ("NN V2","nn_v2"), ("NN V3","nn_v3")]:
    hp = find_history_for_model(m)
    if hp:
        plot_curves_stacked(hp, m, tag)
    else:
        print(f"[WARN] history introuvable pour {m}.")


[OK] NN V1 → /home/a.riyahi/spinn_project/outputs/figs/nn_v1_curves.png
[OK] NN V2 → /home/a.riyahi/spinn_project/outputs/figs/nn_v2_curves.png
[OK] NN V3 → /home/a.riyahi/spinn_project/outputs/figs/nn_v3_curves.png


In [7]:
# 6.0 — Confusions & FN (si preds dispos)
def preds_to_confusion(npz_path):
    D = np.load(npz_path)
    y_true = D.get('y_true_test', D.get('y_true'))
    y_prob = D.get('y_prob_test', D.get('y_prob'))
    if y_true is None or y_prob is None: return None, None, None
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.argmax(y_prob, axis=1) if y_prob.ndim==2 else (y_prob>0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    fn = int((y_true!=y_pred).sum())  # simple total erreurs (ou filtrer classe "damage" si binaire)
    return y_true, y_pred, cm

cms = {}
for m in ["NN V1","NN V2","NN V3"]:
    p = nn_preds.get(m)
    if p and p.exists():
        y_true, y_pred, cm = preds_to_confusion(p)
        if cm is not None:
            cms[m] = cm

# Grille de confusions (si ≥1 dispo)
if cms:
    n = len(cms)
    fig, axes = plt.subplots(1, n, figsize=(3.6*n, 3.2), dpi=300)
    if n==1: axes = [axes]
    for ax, (m, cm) in zip(axes, cms.items()):
        im = ax.imshow(cm, aspect='auto')
        ax.set_title(m); ax.set_xlabel("Pred"); ax.set_ylabel("True")
        for (i,j), v in np.ndenumerate(cm):
            ax.text(j, i, str(v), ha='center', va='center', fontsize=7)
    plt.tight_layout()
    out = FIG_DIR / "confusion_matrices_grid.png"
    plt.savefig(out); plt.close()
    print("→ Grille confusions:", out)
else:
    print("[INFO] Aucune confusion construite (prédictions manquantes).")

# Export FN consolidé (si preds)
fn_rows = []
for m in ["NN V1","NN V2","NN V3"]:
    p = nn_preds.get(m)
    if p and p.exists():
        y_true, y_pred, cm = preds_to_confusion(p)
        if y_true is not None:
            fn_rows.append({"Model":m, "False_Negatives": int((y_true!=y_pred).sum())})
if fn_rows:
    dff = pd.DataFrame(fn_rows)
    dff.to_csv(PKG_DIR/"false_negatives_from_preds.csv", index=False)
    print("→ false_negatives_from_preds.csv")


[INFO] Aucune confusion construite (prédictions manquantes).


In [8]:
# 7.0 — Manifest des sources utilisées
manifest = {
    "metrics_csv": str(nn_metrics) if nn_metrics else None,
    "per_class_csv": str(nn_perclass) if nn_perclass else None,
    "false_negatives_csv": str(nn_fn) if nn_fn else None,
    "predictions": {k: (str(v) if v else None) for k,v in nn_preds.items()},
    "pinn_row_csv": str(pinn_row_csv) if pinn_row_csv else None,
    "pinn_kpi_json": str(pinn_kpi_json) if pinn_kpi_json else None,
    "pinn_diag_npz": str(pinn_diag_npz) if pinn_diag_npz else None,
    "outputs": {
        "table1_csv": str(TAB_DIR/"table1_summary.csv"),
        "table1_md":  str(TAB_DIR/"table1_summary.md"),
        "figs_dir":   str(FIG_DIR),
    }
}
(PKG_DIR/"build_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("→ Manifest:", (PKG_DIR/"build_manifest.json").resolve())
manifest


→ Manifest: /home/a.riyahi/spinn_project/reviewer_pack/build_manifest.json


{'metrics_csv': 'NN/reviewer_pack/metrics_comparison_rounded.csv',
 'per_class_csv': 'NN/reviewer_pack/metrics_per_class_rounded.csv',
 'false_negatives_csv': 'NN/reviewer_pack/false_negatives.csv',
 'predictions': {'NN V1': None, 'NN V2': None, 'NN V3': None},
 'pinn_row_csv': 'reviewer_pack/table1_pinn_row.csv',
 'pinn_kpi_json': 'outputs/artifacts/pinn_kpis_20251110_192717.json',
 'pinn_diag_npz': 'outputs/artifacts/pinn_diag_20251110_192717.npz',
 'outputs': {'table1_csv': '/home/a.riyahi/spinn_project/outputs/tables/table1_summary.csv',
  'table1_md': '/home/a.riyahi/spinn_project/outputs/tables/table1_summary.md',
  'figs_dir': '/home/a.riyahi/spinn_project/outputs/figs'}}

In [9]:
# 7.0 — Packager tous les artefacts pour relecture (ZIP unique)
import os, json, time, hashlib
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

ROOT = Path("/home/a.riyahi/spinn_project") if Path("/home/a.riyahi/spinn_project").exists() else Path.cwd()
PKG  = ROOT / "reviewer_pack"
OUT  = ROOT / "outputs"
FIGS = OUT / "figs"
TABS = OUT / "tables"
PKG.mkdir(parents=True, exist_ok=True)

# 1) Charger le manifest (créé par la cellule précédente)
manifest_path = PKG / "build_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"Manifest introuvable : {manifest_path}")
manifest = json.loads(manifest_path.read_text())

# 2) Candidats à inclure
candidates = set()

def _add(pathlike):
    if not pathlike: return
    p = Path(pathlike)
    if p.exists(): candidates.add(p)

# Depuis le manifest
_add(manifest.get("metrics_csv"))
_add(manifest.get("per_class_csv"))
_add(manifest.get("false_negatives_csv"))

preds = manifest.get("predictions") or {}
for k, v in preds.items():
    _add(v)

_add(manifest.get("pinn_row_csv"))
_add(manifest.get("pinn_kpi_json"))
_add(manifest.get("pinn_diag_npz"))

outs = manifest.get("outputs") or {}
_add(outs.get("table1_csv"))
_add(outs.get("table1_md"))
figs_dir = outs.get("figs_dir")
if figs_dir and Path(figs_dir).exists():
    for p in Path(figs_dir).rglob("*.png"):
        _add(p)

# 3) Ajouter fichiers utiles complémentaires s’ils existent
#    (summary NN, versions brutes, courbes, confusions…)
_add(PKG / "nn_summary_for_table1.csv")
# Ajout de versions non arrondies si présentes
for extra in [
    "NN/reviewer_pack/metrics_comparison.csv",
    "NN/reviewer_pack/metrics_per_class.csv",
    "NN/reviewer_pack/false_negatives.csv",
    "reviewer_pack/table1_pinn_row.csv",
]:
    _add(ROOT / extra)

# Courbes par modèle (si générées)
for name in ["nn_v1_curves.png", "nn_v2_curves.png", "nn_v3_curves.png", "confusion_matrices_grid.png"]:
    p = FIGS / name
    if p.exists(): _add(p)

# Inclure le manifest lui-même
_add(manifest_path)

# 4) Créer un petit README dans le ZIP
readme_txt = f"""JCEF Reviewer Artifacts
========================
Généré: {time.strftime('%Y-%m-%d %H:%M:%S')}
Racine projet: {ROOT}

Contenu principal:
- Table 1 (CSV + Markdown)
- Métriques NN (accuracy, macro-F1, éventuellement ROC_AUC_macro)
- Faux négatifs consolidés
- KPI PINN (résidu h, omega0, omega_est, erreur relative)
- Figures: courbes train/val par modèle (V1/V2/V3), matrices de confusion
- Manifest de traçabilité (build_manifest.json)

NB:
- Si vous exportez ultérieurement les probabilités (y_prob_test) pour NN V1/V2/V3,
  vous pouvez régénérer la colonne ROC_AUC_macro, re-exécuter la fusion et
  re-packager le ZIP.
"""

tmp_readme = PKG / "_JCEF_README.txt"
tmp_readme.write_text(readme_txt, encoding="utf-8")
_add(tmp_readme)

# 5) Construire le ZIP
ts = time.strftime("%Y%m%d_%H%M%S")
zip_path = PKG / f"JCEF_reviewer_artifacts_{ts}.zip"

def arcname_from_root(p: Path) -> str:
    """Nom relatif propre dans le zip (répertoire 'bundle/...' pour clarté)."""
    try:
        rel = p.relative_to(ROOT)
    except Exception:
        rel = p.name
    return str(Path("bundle") / rel)

added, missing = [], []
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED, compresslevel=9) as zf:
    for p in sorted(candidates):
        if p.is_file():
            zf.write(p, arcname=arcname_from_root(p)); added.append(p)
        elif p.is_dir():
            for sub in p.rglob("*"):
                if sub.is_file():
                    zf.write(sub, arcname=arcname_from_root(sub)); added.append(sub)
        else:
            missing.append(p)

# 6) Enlever le README temporaire
try: tmp_readme.unlink(missing_ok=True)
except: pass

# 7) Résumé + hash MD5
def md5sum(path: Path, block=1<<20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

size_mb = zip_path.stat().st_size / (1024*1024)
print(f"[OK] ZIP → {zip_path}  ({size_mb:.2f} MB)")
print(f"[MD5] {md5sum(zip_path)}")
print("\n[INCLUS] Exemples (max 20):")
for p in added[:20]:
    print("  -", p)

if missing:
    print("\n[WARN] Chemins ignorés (ni fichier ni dossier):")
    for p in missing:
        print("  -", p)


[OK] ZIP → /home/a.riyahi/spinn_project/reviewer_pack/JCEF_reviewer_artifacts_20251113_113739.zip  (1.53 MB)
[MD5] bea8611a1193e7ccb2d024aa1ff321be

[INCLUS] Exemples (max 20):
  - /home/a.riyahi/spinn_project/NN/reviewer_pack/false_negatives.csv
  - /home/a.riyahi/spinn_project/NN/reviewer_pack/metrics_comparison.csv
  - /home/a.riyahi/spinn_project/NN/reviewer_pack/metrics_per_class.csv
  - /home/a.riyahi/spinn_project/outputs/figs/nn_v1_curves.png
  - /home/a.riyahi/spinn_project/outputs/figs/nn_v2_curves.png
  - /home/a.riyahi/spinn_project/outputs/figs/nn_v3_curves.png
  - /home/a.riyahi/spinn_project/outputs/figs/paper_psd_20251110_004412.png
  - /home/a.riyahi/spinn_project/outputs/figs/paper_psd_20251110_192715.png
  - /home/a.riyahi/spinn_project/outputs/figs/paper_residual_20251110_004412.png
  - /home/a.riyahi/spinn_project/outputs/figs/paper_residual_20251110_192715.png
  - /home/a.riyahi/spinn_project/outputs/figs/paper_residual_hist_20251110_004412.png
  - /home/a.riyahi/

In [1]:
# 0. EMS manifest ID — auto-detect
from pathlib import Path
import json, hashlib, os

# Adapte si besoin : ton projet est typiquement /home/a.riyahi/spinn_project
ROOT = Path.cwd()
if (ROOT.name != "spinn_project") and (Path("/home/a.riyahi/spinn_project").exists()):
    ROOT = Path("/home/a.riyahi/spinn_project")

# Candidats "classiques" où un manifest peut se trouver
candidates = [
    ROOT/"reviewer_pack/build_manifest.json",
    ROOT/"reviewer_pack/EMS_manifest.json",
    ROOT/"outputs/artifacts/build_manifest.json",
    ROOT/"DatasetPDT/manifest.json",
    ROOT/"data/EMS/manifest.json",
]

# Recherche élargie : tout fichier *manifest*.json du projet
for p in ROOT.rglob("*manifest*.json"):
    if p not in candidates:
        candidates.append(p)

def load_manifest_id(p: Path):
    try:
        d = json.loads(p.read_text())
    except Exception:
        return None, None
    # clés possibles
    keys = ["ems_manifest_id","EMS_manifest_id","manifest_id","ems_id","id"]
    for k in keys:
        if k in d and isinstance(d[k], str) and d[k].strip():
            return d[k].strip(), str(p)
    # recherche imbriquée
    for v in d.values():
        if isinstance(v, dict):
            for k in keys:
                if k in v and isinstance(v[k], str) and v[k].strip():
                    return v[k].strip(), str(p)
    return None, None

ems_id, ems_src = None, None
for p in candidates:
    if p.exists():
        val, src = load_manifest_id(p)
        if val:
            ems_id, ems_src = val, src
            break

print("EMS manifest ID (official):", ems_id)
print("Source:", ems_src)


EMS manifest ID (official): None
Source: None


In [2]:
# 1. Fallback: derive a deterministic EMS ID from your dataset index or file list
import hashlib

DATA_ROOT = ROOT/"DatasetPDT"

# Essayer un index CSV existant
index_csv = None
for p in [
    ROOT/"reviewer_pack/z24_index.csv",
    ROOT/"NN/reviewer_pack/z24_index.csv",
    ROOT/"outputs/hdt/tables/z24_index.csv",
]:
    if p.exists():
        index_csv = p
        break

if index_csv and index_csv.exists():
    raw = index_csv.read_bytes()
    note = f"sha1(z24_index.csv) @ {index_csv}"
elif DATA_ROOT.exists():
    paths = sorted(str(p.relative_to(DATA_ROOT)) for p in DATA_ROOT.rglob("*.mat"))
    raw = ("\n".join(paths)).encode()
    note = f"sha1(list of .mat) under {DATA_ROOT}"
else:
    raw = b""  # rien trouvé (cas extrême)
    note = "no dataset index or .mat list found"

ems_id_fallback = hashlib.sha1(raw).hexdigest()[:16] if raw else None

print("Derived EMS manifest ID (fallback):", ems_id_fallback)
print("Derivation:", note)


Derived EMS manifest ID (fallback): 4b8b4d63c00f3a81
Derivation: sha1(list of .mat) under /home/a.riyahi/spinn_project/DatasetPDT


In [3]:
# 2. Choose final EMS ID and prepare a sentence to paste in the paper
EMS_ID_FINAL = ems_id or ems_id_fallback
assert EMS_ID_FINAL, "No official nor derived EMS ID could be determined."

paper_sentence = f"EMS manifest ID: {EMS_ID_FINAL}."
print("Use in CASE STUDY (first dataset mention):")
print(paper_sentence)


Use in CASE STUDY (first dataset mention):
EMS manifest ID: 4b8b4d63c00f3a81.


In [4]:
# 3. Persist the EMS ID alongside reviewer artifacts (optional)
bm = ROOT/"reviewer_pack/build_manifest.json"
try:
    if bm.exists():
        d = json.loads(bm.read_text())
    else:
        d = {}
    d["ems_manifest_id"] = EMS_ID_FINAL
    bm.parent.mkdir(parents=True, exist_ok=True)
    bm.write_text(json.dumps(d, indent=2))
    print("Updated:", bm)
except Exception as e:
    print("WARN: could not update build_manifest.json ->", e)


Updated: /home/a.riyahi/spinn_project/reviewer_pack/build_manifest.json
